# Arabic Punctuation Restoration

Individual Daal Challenge experiment using AraBERT, multi-label token classification, BCE/Focal model ensembling, and confidence-selected pseudo-labels.

This cleaned version trains both models from scratch (starting from pretrained AraBERT weights), saves checkpoints locally, and produces a competition submission. Training outputs and the original competition score are not embedded in this notebook.


### Metric

In [ ]:
"""
Kaggle metric for Arabic Punctuation Restoration.

Conventions:
    - `solution` is the full test CSV, including the hidden gold column.
    - `submission` is the competitor's CSV.
    - Both have a row_id column (already aligned & sorted by Kaggle).
    - `solution` has a column `raw` with the unpunctuated input and a column
      `gold` with the reference punctuated string.
    - `submission` has a column `prediction` with the competitor's
      punctuated string.

Scoring:
    Macro-F1 over the 7 Arabic sentence-punctuation classes
    ( . ، ؟ ! : ؛ - ), EXCLUDING the "no punctuation" class.

"""

import re
from typing import Optional

import pandas as pd
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer

# ---------------------------------------------------------------------------
# Classical Arabic punctuation-restoration set. Anything outside this is
# treated as word content (e.g. parentheses, quotes, numbers) and is NOT
# scored. Competitors must preserve those characters in their predictions.
# ---------------------------------------------------------------------------
VALID_SYMBOLS = set('.،؟!:؛-')


def _tokenize_gold(text: str):
    """
    Split a (gold or prediction) string into (leading_gap, [(word, trailing_gap), ...]).
    A "word" is a maximal run of non-whitespace, non-whitelist chars.
    A "gap" is any run of whitelist chars between words.
    """
    leading_gap_chars = []
    pairs = []
    current_word_chars = []
    in_word = False

    for ch in text:
        if ch.isspace():
            if in_word:
                pairs.append([''.join(current_word_chars), []])
                current_word_chars = []
                in_word = False
            continue

        if ch in VALID_SYMBOLS:
            if in_word:
                pairs.append([''.join(current_word_chars), [ch]])
                current_word_chars = []
                in_word = False
            else:
                if pairs:
                    pairs[-1][1].append(ch)
                else:
                    leading_gap_chars.append(ch)
            continue

        if not in_word:
            in_word = True
            current_word_chars = [ch]
        else:
            current_word_chars.append(ch)

    if in_word:
        pairs.append([''.join(current_word_chars), []])

    return ''.join(leading_gap_chars), [(w, ''.join(g)) for (w, g) in pairs]


def _extract_labels(raw_text: str, generated_text: str, role: str):
    """
    Align `generated_text` to `raw_text` word-by-word and return one list of
    symbols per word, representing the punctuation that appears in the gap
    after each word.

    Raises ValueError on any structural mismatch.
    """
    if raw_text is None or generated_text is None:
        raise ValueError(f"[{role}] text is empty or null")

    raw_words = str(raw_text).strip().split()
    if not raw_words:
        raise ValueError(f"[{role}] raw text contains no words")

    _, pairs = _tokenize_gold(str(generated_text))
    gen_words = [w for (w, _) in pairs]

    if len(gen_words) != len(raw_words):
        raise ValueError(
            f"[{role}] word-count mismatch: raw has {len(raw_words)} words, "
            f"{role} has {len(gen_words)}"
        )

    for i, (rw, gw) in enumerate(zip(raw_words, gen_words)):
        if rw != gw:
            raise ValueError(
                f"[{role}] word mismatch at position {i}: "
                f"raw='{rw}' vs {role}='{gw}'"
            )

    labels = []
    for _, gap in pairs:
        syms = [c for c in gap if c in VALID_SYMBOLS]
        labels.append(syms if syms else ['0'])

    return labels


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str,
    raw_column_name: str = 'text',
    gold_column_name: str = 'final_text',
    prediction_column_name: str = 'final_text',
) -> float:
    """
    Returns macro-F1 over the 7 Arabic punctuation
    classes, excluding the "no punctuation" class.
    """
    # --- 0. Drop row_id; Kaggle has already aligned the two frames -----------
    del solution[row_id_column_name]
    del submission[row_id_column_name]

    # --- 1. Column presence -------------------------------------------------
    for col in (raw_column_name, gold_column_name):
        if col not in solution.columns:
            # Organizer-side problem — hidden from competitor.
            raise RuntimeError(f"Solution is missing column '{col}'")
    if prediction_column_name not in submission.columns:
        raise ParticipantVisibleError(
            f"Submission is missing column '{prediction_column_name}'"
        )

    # --- 2. Length match ----------------------------------------------------
    if len(solution) != len(submission):
        raise ParticipantVisibleError(
            f"Submission has {len(submission)} rows, expected {len(solution)}"
        )

    # --- 3. Extract labels row-by-row --------------------------------------
    true_labels = []
    pred_labels = []

    raws = solution[raw_column_name].tolist()
    golds = solution[gold_column_name].tolist()
    preds = submission[prediction_column_name].tolist()

    for idx, (raw, gold, pred) in enumerate(zip(raws, golds, preds)):
        try:
            gold_seq = _extract_labels(raw, gold, role="gold")
        except ValueError as e:
            # Organizer-side: our own gold CSV is malformed for this row.
            raise RuntimeError(
                f"Gold extraction failed on row index {idx}: {e}"
            ) from e

        try:
            pred_seq = _extract_labels(raw, pred, role="prediction")
        except ValueError as e:
            # Competitor can fix this themselves.
            raise ParticipantVisibleError(
                f"Prediction at row index {idx} does not align with the raw "
                f"input. Your prediction must contain the same sequence of "
                f"non-punctuation words as the input, with only the allowed "
                f"punctuation symbols {sorted(VALID_SYMBOLS)} inserted "
                f"between them. Details: {e}"
            ) from e

        if len(gold_seq) != len(pred_seq):
            raise ParticipantVisibleError(
                f"Row {idx}: prediction has {len(pred_seq)} word positions "
                f"but the input has {len(gold_seq)}"
            )

        true_labels.extend(gold_seq)
        pred_labels.extend(pred_seq)

    # --- 4. Binarize using gold ∪ pred so hallucinated marks cost precision --
    all_classes = sorted(VALID_SYMBOLS) + ['0']
    mlb = MultiLabelBinarizer(classes=all_classes)
    y_true = mlb.fit_transform(true_labels)
    y_pred = mlb.transform(pred_labels)

    classes = list(mlb.classes_)
    zero_idx = classes.index('0')
    scored_cols = [i for i in range(len(classes)) if i != zero_idx]

    return float(f1_score(
        y_true[:, scored_cols],
        y_pred[:, scored_cols],
        average='macro',
        zero_division=0,
    ))

## 1. Setup

In [ ]:
from IPython.display import display
import os
import random
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)
import datasets as hfds

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL_NAME = "aubmindlab/bert-base-arabertv02"
TRAIN_CSV  = "train.csv"
TEST_CSV   = "test.csv"
OUTPUT_DIR = "./camelbert-punct"
SUBMISSION_CSV = "submission.csv"

MAX_LEN   = 384
BATCH     = 8
EPOCHS    = 8
LR        = 2e-5

# The 7 scored punctuation marks plus '0' for "no punctuation"
# لا يوجد كلاس 0 هنا؛ عدم وجود ترقيم = كل القيم صفر
LABELS = [".", "،", "؟", "!", ":", "؛", "-"]

LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}
NUM_LABELS = len(LABELS)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "| Model:", MODEL_NAME)

## 2. Load training data

This experiment trains on all provided training rows. It does not create a held-out validation split. Test labels are unavailable.


In [ ]:
train_df = pd.read_csv(TRAIN_CSV).reset_index(drop=True)

print(f"Training rows: {len(train_df)}")
print(train_df.head(2).to_string())

## 3. Create word-level multi-label targets

Each word receives a seven-element vector for `. ، ؟ ! : ؛ -`. Multiple marks can follow the same word; no punctuation is represented by an all-zero vector. The tokenizer follows the supplied competition metric.


In [ ]:
VALID = set(LABELS)
  # the 7 punctuation chars

def _tokenize_with_gaps(text):
    """Walk the string and split into (word, trailing_gap) pairs.
    This matches the competition metric's tokenizer exactly."""
    pairs = []
    cur = []
    in_word = False
    for ch in str(text):
        if ch.isspace():
            if in_word:
                pairs.append([''.join(cur), []])
                cur = []
                in_word = False
            continue
        if ch in VALID:
            if in_word:
                pairs.append([''.join(cur), [ch]])
                cur = []
                in_word = False
            else:
                if pairs:
                    pairs[-1][1].append(ch)
            continue
        if not in_word:
            in_word = True
            cur = [ch]
        else:
            cur.append(ch)
    if in_word:
        pairs.append([''.join(cur), []])
    return [(w, ''.join(g)) for (w, g) in pairs]

def split_to_words_and_labels(final_text):
    """
    لكل كلمة ننتج vector بطول 7.
    مثال ؟! يصبح:
    [0, 0, 1, 1, 0, 0, 0]
    """
    pairs = _tokenize_with_gaps(final_text)

    words = [word for word, _ in pairs]
    labels = []

    for _, gap in pairs:
        multi_hot = [0.0] * NUM_LABELS

        for symbol in gap:
            if symbol in LABEL2ID:
                multi_hot[LABEL2ID[symbol]] = 1.0

        labels.append(multi_hot)

    return words, labels

# Sanity check
ex_words, ex_labels = split_to_words_and_labels(train_df["final_text"].iloc[0])
print("First 12 words:", ex_words[:12])
print("First 12 labels:", [i for i in ex_labels[:12]])

## 4. Tokenization and label alignment

Long texts are split into overlapping windows. Labels are assigned to the last subtoken of each word; other subtokens and special tokens are masked with -100. Inference also reads the last subtoken.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(words_batch, labels_batch=None):
    enc = tokenizer(
        words_batch,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
    )

    if labels_batch is None:
        return enc

    aligned = []

    for batch_index, word_labels in enumerate(labels_batch):
        word_ids = enc.word_ids(batch_index=batch_index)

        token_labels = [
            [-100.0] * NUM_LABELS
            for _ in word_ids
        ]

        # نحدد آخر subtoken لكل كلمة
        last_token_for_word = {}

        for token_index, word_id in enumerate(word_ids):
            if word_id is not None:
                last_token_for_word[word_id] = token_index

        # نضع label على آخر subtoken فقط
        for word_id, token_index in last_token_for_word.items():
            if word_id < len(word_labels):
                token_labels[token_index] = word_labels[word_id]

        aligned.append(token_labels)

    enc["labels"] = aligned
    return enc


def split_into_token_windows(words, labels, max_len=MAX_LEN, overlap_words=40):
    """
    تقسيم النص الطويل إلى أجزاء لا تتجاوز MAX_LEN tokens،
    مع تداخل بين الأجزاء للمحافظة على السياق.
    """
    chunks_words = []
    chunks_labels = []

    start = 0
    n_words = len(words)

    while start < n_words:
        end = start
        current_token_count = 2  # [CLS] و [SEP]

        while end < n_words:
            word_tokens = tokenizer.tokenize(words[end])
            word_token_count = max(len(word_tokens), 1)

            if current_token_count + word_token_count > max_len:
                break

            current_token_count += word_token_count
            end += 1

        # احتياط لو كانت كلمة واحدة طويلة جدًا
        if end == start:
            end = start + 1

        chunks_words.append(words[start:end])
        chunks_labels.append(labels[start:end])

        if end >= n_words:
            break

        # نرجع عددًا من الكلمات للخلف حتى يكون هناك سياق مشترك
        start = max(start + 1, end - overlap_words)

    return chunks_words, chunks_labels


def build_dataset(df):
    all_words = []
    all_labels = []

    for _, row in df.iterrows():
        words, labels = split_to_words_and_labels(row["final_text"])

        if not words:
            continue

        word_chunks, label_chunks = split_into_token_windows(
            words,
            labels,
            max_len=MAX_LEN,
            overlap_words=40,
        )

        all_words.extend(word_chunks)
        all_labels.extend(label_chunks)

    return hfds.Dataset.from_dict({
        "words": all_words,
        "labels": all_labels,
    })

train_ds = build_dataset(train_df)

def tokenize_batch(batch):
    return encode(batch["words"], batch["labels"])

train_tok = train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_ds.column_names,
)

print(train_tok)

In [ ]:
import torch

positive_counts = torch.zeros(NUM_LABELS)
total_positions = 0

for row in train_tok:
    labels = torch.tensor(row["labels"], dtype=torch.float32)

    valid_mask = labels[:, 0] != -100
    valid_labels = labels[valid_mask]

    positive_counts += valid_labels.sum(dim=0)
    total_positions += valid_labels.shape[0]

negative_counts = total_positions - positive_counts

pos_weights = negative_counts / positive_counts.clamp(min=1)

pos_weights = torch.clamp(
    pos_weights,
    min=1.0,
    max=20.0,
)

print("Positive counts:", positive_counts)
print("Positive weights:", pos_weights)

In [ ]:
class MultiLabelTokenCollator:
    def __init__(self, tokenizer, num_labels):
        self.tokenizer = tokenizer
        self.num_labels = num_labels

    def __call__(self, features):
        # نفصل labels قبل padding
        labels = [feature.pop("labels") for feature in features]

        # نعمل padding لبقية المدخلات
        batch = self.tokenizer.pad(
            features,
            padding=True,
            return_tensors="pt",
        )

        max_length = batch["input_ids"].shape[1]

        padded_labels = []

        for sequence_labels in labels:
            sequence_labels = [
                list(label) for label in sequence_labels
            ]

            padding_length = max_length - len(sequence_labels)

            sequence_labels.extend(
                [[-100.0] * self.num_labels] * padding_length
            )

            padded_labels.append(sequence_labels)

        batch["labels"] = torch.tensor(
            padded_labels,
            dtype=torch.float32,
        )

        return batch

In [ ]:
multi_label_collator = MultiLabelTokenCollator(
    tokenizer=tokenizer,
    num_labels=NUM_LABELS,
)

## 5. Train BCE and Focal models

Two AraBERT models are trained on the full training dataset. This notebook does not calculate held-out performance during training. The supplied competition metric computes macro-F1 over seven punctuation classes, excluding no-punctuation positions as a class.


In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    problem_type="multi_label_classification",
)

args = TrainingArguments(
    output_dir="./final_bce_full",

    learning_rate=LR,
    per_device_train_batch_size=BATCH,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,

    eval_strategy="no",
    save_strategy="epoch",
    save_total_limit=8,
    load_best_model_at_end=False,

    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
    seed=SEED,
)


class MultiLabelTrainer(Trainer):
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        **kwargs,
    ):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        logits = outputs.logits

        valid_mask = labels[:, :, 0] != -100

        valid_logits = logits[valid_mask]
        valid_labels = labels[valid_mask]

        loss_function = torch.nn.BCEWithLogitsLoss(
            pos_weight=pos_weights.to(logits.device),
        )

        loss = loss_function(
            valid_logits,
            valid_labels,
        )

        return (loss, outputs) if return_outputs else loss


trainer = MultiLabelTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    data_collator=multi_label_collator,
)

trainer.train()

In [ ]:
LAST_BCE_PATH = "./models/last_subtoken_bce_full"
LAST_FOCAL_PATH = "./models/last_subtoken_focal_full"

trainer.save_model(LAST_BCE_PATH)
tokenizer.save_pretrained(LAST_BCE_PATH)
print("Saved BCE model:", LAST_BCE_PATH)


In [ ]:
class FocalMultiLabelTrainer(Trainer):
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        **kwargs,
    ):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        logits = outputs.logits

        valid_mask = labels[:, :, 0] != -100

        valid_logits = logits[valid_mask]
        valid_labels = labels[valid_mask]

        bce_loss = torch.nn.functional.binary_cross_entropy_with_logits(
            valid_logits,
            valid_labels,
            pos_weight=pos_weights.to(logits.device),
            reduction="none",
        )

        probabilities = torch.sigmoid(valid_logits)

        pt = torch.where(
            valid_labels == 1,
            probabilities,
            1 - probabilities,
        )

        gamma = 2.0
        focal_factor = (1 - pt) ** gamma

        loss = (focal_factor * bce_loss).mean()

        return (loss, outputs) if return_outputs else loss


last_focal_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    problem_type="multi_label_classification",
)

last_focal_args = TrainingArguments(
    output_dir="./last_subtoken_focal_full",
    learning_rate=LR,
    per_device_train_batch_size=BATCH,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy="no",
    save_strategy="epoch",
    save_total_limit=8,
    load_best_model_at_end=False,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

last_focal_trainer = FocalMultiLabelTrainer(
    model=last_focal_model,
    args=last_focal_args,
    train_dataset=train_tok,
    data_collator=multi_label_collator,
)

last_focal_trainer.train()

In [ ]:
LAST_FOCAL_PATH = "./models/last_subtoken_focal_full"

last_focal_trainer.save_model(LAST_FOCAL_PATH)
tokenizer.save_pretrained(LAST_FOCAL_PATH)

print("Saved:", LAST_FOCAL_PATH)

In [ ]:
from transformers import AutoModelForTokenClassification

LAST_BCE_PATH = "./models/last_subtoken_bce_full"
LAST_FOCAL_PATH = "./models/last_subtoken_focal_full"

last_bce_model = AutoModelForTokenClassification.from_pretrained(
    LAST_BCE_PATH
).to(device).eval()

last_focal_model = AutoModelForTokenClassification.from_pretrained(
    LAST_FOCAL_PATH
).to(device).eval()

print("تم تحميل موديلات Last Subtoken")

In [ ]:
import time

pseudo_start_time = time.time()

test_df = pd.read_csv(TEST_CSV).reset_index(drop=True)

print("Test rows:", len(test_df))
print(test_df.head(2))

In [ ]:
LAST_BCE_WEIGHT = 0.51
LAST_FOCAL_WEIGHT = 0.49

LAST_THRESHOLDS = torch.tensor(
    [0.82, 0.75, 0.82, 0.76, 0.81, 0.80, 0.87],
    dtype=torch.float32,
)

@torch.no_grad()
def predict_punctuated_last_subtoken(text, overlap_words=20):
    words = str(text).split()

    if not words:
        return ""

    bce_logits_sum = torch.zeros(
        len(words), NUM_LABELS, dtype=torch.float32
    )
    focal_logits_sum = torch.zeros(
        len(words), NUM_LABELS, dtype=torch.float32
    )
    word_counts = torch.zeros(len(words), dtype=torch.float32)

    start = 0

    while start < len(words):
        end = start
        token_count = 2

        while end < len(words):
            word_token_count = max(
                len(tokenizer.tokenize(words[end])), 1
            )

            if token_count + word_token_count > MAX_LEN:
                break

            token_count += word_token_count
            end += 1

        if end == start:
            end = start + 1

        chunk = words[start:end]

        enc = tokenizer(
            chunk,
            is_split_into_words=True,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LEN,
        )

        model_inputs = {
            key: value.to(device)
            for key, value in enc.items()
        }

        bce_logits = last_bce_model(
            **model_inputs
        ).logits[0].cpu()

        focal_logits = last_focal_model(
            **model_inputs
        ).logits[0].cpu()

        word_ids = enc.word_ids(batch_index=0)

        # نأخذ آخر subtoken لكل كلمة
        last_token_for_word = {}

        for token_index, word_id in enumerate(word_ids):
            if word_id is not None:
                last_token_for_word[word_id] = token_index

        for word_id, token_index in last_token_for_word.items():
            global_index = start + word_id

            bce_logits_sum[global_index] += bce_logits[token_index]
            focal_logits_sum[global_index] += focal_logits[token_index]
            word_counts[global_index] += 1

        if end >= len(words):
            break

        start = max(start + 1, end - overlap_words)

    pieces = []

    for index, word in enumerate(words):
        if word_counts[index] == 0:
            pieces.append(word)
            continue

        avg_bce_logits = (
            bce_logits_sum[index] / word_counts[index]
        )
        avg_focal_logits = (
            focal_logits_sum[index] / word_counts[index]
        )

        bce_probs = torch.sigmoid(avg_bce_logits)
        focal_probs = torch.sigmoid(avg_focal_logits)

        final_probs = (
            LAST_BCE_WEIGHT * bce_probs
            + LAST_FOCAL_WEIGHT * focal_probs
        )

        selected_symbols = []

        for label_id, probability in enumerate(final_probs):
            if probability >= LAST_THRESHOLDS[label_id]:
                selected_symbols.append(ID2LABEL[label_id])

        pieces.append(word + "".join(selected_symbols))

    return " ".join(pieces)

In [ ]:
@torch.no_grad()
def predict_pseudo_with_confidence(text, overlap_words=120):
    words = str(text).split()

    if not words:
        return "", 0.0

    bce_logits_sum = torch.zeros(
        len(words), NUM_LABELS, dtype=torch.float32
    )

    focal_logits_sum = torch.zeros(
        len(words), NUM_LABELS, dtype=torch.float32
    )

    word_counts = torch.zeros(
        len(words), dtype=torch.float32
    )

    start = 0

    while start < len(words):
        end = start
        token_count = 2

        while end < len(words):
            word_token_count = max(
                len(tokenizer.tokenize(words[end])),
                1,
            )

            if token_count + word_token_count > MAX_LEN:
                break

            token_count += word_token_count
            end += 1

        if end == start:
            end = start + 1

        chunk = words[start:end]

        enc = tokenizer(
            chunk,
            is_split_into_words=True,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LEN,
        )

        model_inputs = {
            key: value.to(device)
            for key, value in enc.items()
        }

        bce_logits = last_bce_model(
            **model_inputs
        ).logits[0].float().cpu()

        focal_logits = last_focal_model(
            **model_inputs
        ).logits[0].float().cpu()

        word_ids = enc.word_ids(batch_index=0)

        last_token_for_word = {}

        for token_index, word_id in enumerate(word_ids):
            if word_id is not None:
                last_token_for_word[word_id] = token_index

        for word_id, token_index in last_token_for_word.items():
            global_index = start + word_id

            if global_index >= len(words):
                continue

            bce_logits_sum[global_index] += (
                bce_logits[token_index]
            )

            focal_logits_sum[global_index] += (
                focal_logits[token_index]
            )

            word_counts[global_index] += 1

        if end >= len(words):
            break

        start = max(
            start + 1,
            end - overlap_words,
        )

    pieces = []
    decision_confidences = []
    agreement_scores = []

    for index, word in enumerate(words):
        if word_counts[index] == 0:
            pieces.append(word)
            continue

        avg_bce_logits = (
            bce_logits_sum[index]
            / word_counts[index]
        )

        avg_focal_logits = (
            focal_logits_sum[index]
            / word_counts[index]
        )

        bce_probs = torch.sigmoid(avg_bce_logits)
        focal_probs = torch.sigmoid(avg_focal_logits)

        final_probs = (
            LAST_BCE_WEIGHT * bce_probs
            + LAST_FOCAL_WEIGHT * focal_probs
        )

        selected_symbols = []
        selected_confidences = []

        for label_id, probability in enumerate(final_probs):
            threshold = float(LAST_THRESHOLDS[label_id])
            probability_value = float(probability)

            if probability_value >= threshold:
                selected_symbols.append(
                    ID2LABEL[label_id]
                )

                positive_confidence = (
                    probability_value - threshold
                ) / max(
                    1.0 - threshold,
                    1e-6,
                )

                selected_confidences.append(
                    positive_confidence
                )

        if selected_confidences:
            decision_confidences.extend(
                selected_confidences
            )

        agreement = 1.0 - torch.abs(
            bce_probs - focal_probs
        ).mean().item()

        agreement_scores.append(
            agreement
        )

        pieces.append(
            word + "".join(selected_symbols)
        )

    mean_decision_confidence = (
        float(np.mean(decision_confidences))
        if decision_confidences
        else 0.0
    )

    punctuation_density = (
        len(decision_confidences)
        / max(len(words), 1)
    )

    density_score = min(
        punctuation_density / 0.08,
        1.0,
    )

    mean_decision_confidence *= density_score

    mean_agreement = (
        float(np.mean(agreement_scores))
        if agreement_scores
        else 0.0
    )

    row_confidence = (
        0.85 * mean_decision_confidence
        + 0.15 * mean_agreement
    )

    return " ".join(pieces), row_confidence

In [ ]:
sample_prediction, sample_confidence = (
    predict_pseudo_with_confidence(
        test_df["text"].iloc[0]
    )
)

print("Confidence:", sample_confidence)
print("\nPrediction:")
print(sample_prediction[:1000])

In [ ]:
from tqdm.auto import tqdm
import time

pseudo_predictions = []
pseudo_confidences = []

start_time = time.time()

for text in tqdm(
    test_df["text"].tolist(),
    desc="Generating pseudo labels"
):
    prediction, confidence = predict_pseudo_with_confidence(text)

    pseudo_predictions.append(prediction)
    pseudo_confidences.append(confidence)

test_pseudo_df = test_df.copy()
test_pseudo_df["pseudo_final_text"] = pseudo_predictions
test_pseudo_df["pseudo_confidence"] = pseudo_confidences

elapsed_minutes = (time.time() - start_time) / 60

print(f"Finished in {elapsed_minutes:.2f} minutes")
print(test_pseudo_df["pseudo_confidence"].describe())

test_pseudo_df.to_csv(
    "test_pseudo_with_confidence.csv",
    index=False,
)

print("Saved: test_pseudo_with_confidence.csv")

In [ ]:
for q in [0.50, 0.60, 0.70, 0.75, 0.80, 0.90]:
    print(
        f"Quantile {q:.2f}:",
        test_pseudo_df["pseudo_confidence"].quantile(q)
    )

In [ ]:
display(
    test_pseudo_df[
        ["text", "pseudo_final_text", "pseudo_confidence"]
    ]
    .sort_values(
        "pseudo_confidence",
        ascending=False,
    )
    .head(5)
)

display(
    test_pseudo_df[
        ["text", "pseudo_final_text", "pseudo_confidence"]
    ]
    .sort_values(
        "pseudo_confidence",
        ascending=True,
    )
    .head(5)
)

In [ ]:
PSEUDO_QUANTILE = 0.80

pseudo_cutoff = test_pseudo_df[
    "pseudo_confidence"
].quantile(PSEUDO_QUANTILE)

selected_pseudo = (
    test_pseudo_df[
        test_pseudo_df["pseudo_confidence"] >= pseudo_cutoff
    ]
    .copy()
    .reset_index(drop=True)
)

print("Confidence cutoff:", pseudo_cutoff)
print("Selected pseudo rows:", len(selected_pseudo))
print(
    "Selected confidence range:",
    selected_pseudo["pseudo_confidence"].min(),
    "to",
    selected_pseudo["pseudo_confidence"].max(),
)

In [ ]:
pseudo_train_df = pd.DataFrame({
    "final_text": selected_pseudo["pseudo_final_text"]
})

print(pseudo_train_df.shape)
print(pseudo_train_df.head(2).to_string())

In [ ]:
real_part = train_df[["final_text"]].copy()
real_part["data_source"] = "real"

pseudo_part = pseudo_train_df[["final_text"]].copy()
pseudo_part["data_source"] = "pseudo"

combined_train_df = pd.concat(
    [real_part, pseudo_part],
    ignore_index=True,
)

combined_train_df = combined_train_df.sample(
    frac=1.0,
    random_state=42,
).reset_index(drop=True)

print("Original training rows:", len(real_part))
print("Pseudo rows:", len(pseudo_part))
print("Combined rows:", len(combined_train_df))

print("\nSource counts:")
print(combined_train_df["data_source"].value_counts())

In [ ]:
combined_train_ds = build_dataset(
    combined_train_df[["final_text"]]
)

combined_train_tok = combined_train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=combined_train_ds.column_names,
)

print(combined_train_ds)
print(combined_train_tok)

In [ ]:
pseudo_positive_counts = torch.zeros(NUM_LABELS)
pseudo_total_positions = 0

for row in combined_train_tok:
    labels = torch.tensor(
        row["labels"],
        dtype=torch.float32,
    )

    valid_mask = labels[:, 0] != -100
    valid_labels = labels[valid_mask]

    pseudo_positive_counts += valid_labels.sum(dim=0)
    pseudo_total_positions += valid_labels.shape[0]

pseudo_negative_counts = (
    pseudo_total_positions - pseudo_positive_counts
)

pseudo_pos_weights = (
    pseudo_negative_counts
    / pseudo_positive_counts.clamp(min=1)
)

pseudo_pos_weights = torch.clamp(
    pseudo_pos_weights,
    min=1.0,
    max=20.0,
)

print("Positive counts:", pseudo_positive_counts)
print("Positive weights:", pseudo_pos_weights)

In [ ]:
from transformers import TrainingArguments

PSEUDO_BCE_OUTPUT = "./pseudo_bce_finetuned"

# نحمّل نسخة جديدة من مودل BCE المحفوظ
pseudo_bce_model = AutoModelForTokenClassification.from_pretrained(
    LAST_BCE_PATH
)

pseudo_bce_args = TrainingArguments(
    output_dir=PSEUDO_BCE_OUTPUT,

    # Fine-tuning خفيف حتى لا ينسى النموذج التدريب الأصلي
    learning_rate=5e-6,
    per_device_train_batch_size=BATCH,
    num_train_epochs=2,
    weight_decay=0.01,

    eval_strategy="no",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=False,

    fp16=torch.cuda.is_available(),
    logging_steps=20,
    report_to="none",
    seed=SEED,
)

In [ ]:
class PseudoBCETrainer(Trainer):
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        **kwargs,
    ):
        labels = inputs.pop("labels").float()

        outputs = model(**inputs)
        logits = outputs.logits

        valid_mask = labels[:, :, 0] != -100

        valid_logits = logits[valid_mask]
        valid_labels = labels[valid_mask]

        loss_function = torch.nn.BCEWithLogitsLoss(
            pos_weight=pseudo_pos_weights.to(logits.device)
        )

        loss = loss_function(
            valid_logits,
            valid_labels,
        )

        return (loss, outputs) if return_outputs else loss

In [ ]:
pseudo_bce_trainer = PseudoBCETrainer(
    model=pseudo_bce_model,
    args=pseudo_bce_args,
    train_dataset=combined_train_tok,
    data_collator=multi_label_collator,
)

pseudo_bce_trainer.train()

In [ ]:
PSEUDO_BCE_SAVE_PATH = (
    "./models/"
    "pseudo_bce_top20_finetuned"
)

pseudo_bce_trainer.save_model(
    PSEUDO_BCE_SAVE_PATH
)

tokenizer.save_pretrained(
    PSEUDO_BCE_SAVE_PATH
)

print("Saved:", PSEUDO_BCE_SAVE_PATH)

In [ ]:
last_bce_model = AutoModelForTokenClassification.from_pretrained(
    PSEUDO_BCE_SAVE_PATH
).to(device).eval()

# مودل Focal يبقى كما هو بدون إعادة تدريب
last_focal_model = AutoModelForTokenClassification.from_pretrained(
    LAST_FOCAL_PATH
).to(device).eval()

print("Loaded fine-tuned BCE + original Focal")

In [ ]:
from tqdm.auto import tqdm

new_predictions = [
    predict_punctuated_last_subtoken(text)
    for text in tqdm(
        test_df["text"].tolist(),
        desc="Generating final submission"
    )
]

submission_pseudo = pd.DataFrame({
    "id": test_df["id"],
    "final_text": new_predictions,
})

submission_pseudo.to_csv(
    "submission_overlap40.csv",
    index=False,
)

print("Saved: submission_overlap40.csv")
print("Rows:", len(submission_pseudo))

submission_pseudo.head()

In [ ]:
def check_submission_alignment(raw_text, predicted_text):
    raw_words = str(raw_text).split()

    _, predicted_pairs = _tokenize_gold(
        str(predicted_text)
    )

    predicted_words = [
        word
        for word, _ in predicted_pairs
    ]

    return raw_words == predicted_words


alignment_results = [
    check_submission_alignment(raw, pred)
    for raw, pred in zip(
        test_df["text"],
        submission_pseudo["final_text"],
    )
]

print("Valid rows:", sum(alignment_results))
print("Invalid rows:", len(alignment_results) - sum(alignment_results))

In [ ]:
print("Submission is available at: submission_overlap40.csv")
